In [ ]:
# ── Path setup ───────────────────────────────────────────────────────────
# All paths are derived from BASE_DIR. Override CAPSTONE_BASE_DIR if you
# want artifacts written somewhere other than the project root.
import os, sys

BASE_DIR  = os.environ.get("CAPSTONE_BASE_DIR", os.getcwd())
MODEL_DIR = os.path.join(BASE_DIR, "trained_model")
DATA_DIR  = os.path.join(BASE_DIR, "data")
CKPT_DIR  = os.path.join(BASE_DIR, "checkpoints")
DATA_CSV  = os.path.join(DATA_DIR, "paysim.csv")

for d in (MODEL_DIR, DATA_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, BASE_DIR)
print(f"BASE_DIR = {BASE_DIR}")
print(f"  Data CSV expected at: {DATA_CSV}")

## Install dependencies

In [ ]:
# Heavy hitters live here so the import cell below is fast.
!pip install -q transformers scikit-learn torch scipy

## Imports

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import average_precision_score, f1_score
import pickle, json, argparse
from pathlib import Path
from scipy.special import expit
import importlib

# model_components is the sibling module that defines the architecture
# and the dataset wrapper. We reload it explicitly so notebook edits to
# that file get picked up without restarting the kernel.
import model_components
importlib.reload(model_components)

from model_components import (
    LongRangeFraudTransformer,
    engineer_behavioral_features,
    PaySimSequenceDataset,
    FEATURE_COLS,
    TYPE_COL_IDX, NUM_TYPE_CATS, TYPE_EMB_DIM,
)
print("✓ model_components reloaded")

## Focal Loss, Metrics & Wrapped Model

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FOCAL LOSS
# ══════════════════════════════════════════════════════════════════════
# Standard BCE drowns the rare-fraud signal because of the heavy class
# imbalance (~0.97% fraud). Focal loss reshapes the loss curve so the
# model spends most of its capacity on the hard / rare examples.

def focal_loss_with_logits(logits, targets, gamma=2.0, pos_weight=None):
    """Focal binary cross-entropy: ((1 - p_t)**gamma) * BCE."""
    bce   = F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=pos_weight, reduction="none"
    )
    pt    = torch.exp(-bce)              # probability assigned to true class
    focal = ((1.0 - pt) ** gamma) * bce
    return focal.mean()


# ══════════════════════════════════════════════════════════════════════
# METRICS — AUPRC and F1, computed every epoch by the HF Trainer
# ══════════════════════════════════════════════════════════════════════
# AUPRC is a more honest signal than ROC-AUC under heavy class imbalance,
# so we use it as the early-stopping / best-model criterion.

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = logits.squeeze(-1)
    probs  = expit(logits)   # sigmoid

    try:
        auprc = average_precision_score(labels, probs)
    except ValueError:
        auprc = 0.0

    preds = (probs >= 0.5).astype(int)
    f1    = f1_score(labels, preds, zero_division=0)

    return {"auprc": auprc, "f1": f1}


# ══════════════════════════════════════════════════════════════════════
# WRAPPED MODEL — adapter so HF Trainer can drive our custom model
# ══════════════════════════════════════════════════════════════════════
# The HF Trainer expects a `loss` field in the forward output; our model
# returns its own auxiliary loss instead. This wrapper combines them and
# applies focal loss to the anomaly head.

class WrappedModel(nn.Module):
    def __init__(self, m, fraud_weight: float = 10.0, focal_gamma: float = 2.0):
        super().__init__()
        self.model        = m
        self.fraud_weight = fraud_weight
        self.focal_gamma  = focal_gamma

    def forward(
        self,
        past_values,
        past_time_features,
        past_observed_mask,
        future_values,
        future_time_features,
        labels=None,
        **kwargs,
    ):
        out  = self.model(past_values, past_time_features, future_values)
        # Down-weight the reconstruction term — it's a regulariser, not
        # the primary objective.
        loss = 0.2 * out["loss"]

        if labels is not None:
            logits = out["anomaly_score"].squeeze(-1)
            pw     = torch.tensor(
                self.fraud_weight, dtype=torch.float32, device=logits.device
            )
            focal  = focal_loss_with_logits(
                logits, labels.float(),
                gamma=self.focal_gamma,
                pos_weight=pw,
            )
            loss = loss + focal

        return {
            "loss":   loss,
            "logits": out["anomaly_score"],   # raw logits — compute_metrics applies sigmoid
        }


print("✓ Focal loss, metrics, and WrappedModel defined.")

## Training function

In [ ]:
def train(data_path=DATA_CSV, output_dir=MODEL_DIR, epochs=20, context_len=10):
    """Full training pipeline: load → split → resample → fit → save."""
    print(f"Loading training data from {data_path}...")
    df = pd.read_csv(data_path)

    print(f"Raw shape:    {df.shape}")
    print(f"Columns:      {df.columns.tolist()}")
    print(f"Type values:  {df['type'].value_counts().to_dict() if 'type' in df.columns else 'NO type COLUMN'}")

    # Restrict to the two transaction types where fraud actually occurs.
    df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

    # Encode `type` and engineer the behavioural features used downstream.
    le = LabelEncoder()
    df['type_enc'] = le.fit_transform(df['type'])
    df = df.sort_values(by=['nameOrig', 'step']).reset_index(drop=True)
    df = engineer_behavioral_features(df)
    df = df.sort_values('step').reset_index(drop=True)
    print(f"Filtered data: {len(df):,} transactions")

    # Time-based 70 / 15 / 15 split — never use random splitting on
    # transactional data because it leaks future info into the past.
    n     = len(df)
    t_end = int(n * 0.70)
    v_end = int(n * 0.85)

    train_df = df.iloc[:t_end].copy()
    val_df   = df.iloc[t_end:v_end].copy()
    test_df  = df.iloc[v_end:].copy()

    print(f"Split (rows) — Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
    print(f"Fraud rate   — Train: {train_df['isFraud'].mean():.4f} | "
          f"Val: {val_df['isFraud'].mean():.4f} | Test: {test_df['isFraud'].mean():.4f}")

    # Down-sample the negatives to a 1:10 fraud:normal ratio. Combined
    # with focal loss, this gives the model a fighting chance on the
    # rare positive class without throwing away the time ordering.
    fraud_df       = train_df[train_df['isFraud'] == 1]
    normal_df      = train_df[train_df['isFraud'] == 0]
    normal_sampled = resample(normal_df, n_samples=len(fraud_df) * 10, random_state=42)
    train_df = pd.concat([fraud_df, normal_sampled]).sort_values('step').reset_index(drop=True)
    print(f"\nResampled train: {len(train_df):,} rows | "
          f"Fraud: {train_df['isFraud'].sum():,} | Normal: {(train_df['isFraud']==0).sum():,}")

    # Persist the test split so deploy_model can reuse exactly the same rows.
    test_csv_path = os.path.join(DATA_DIR, "test_data.csv")
    test_df.to_csv(test_csv_path, index=False)
    print(f"\n✓ Test data saved → {test_csv_path}  ({len(test_df):,} rows)")

    # Build the sliding-window datasets. Stride=1 on training data
    # maximises the number of fraud-containing windows we ever see.
    STRIDE   = 1
    train_ds = PaySimSequenceDataset(
        train_df, context_len=context_len, train=True,
        feature_cols=FEATURE_COLS, stride=STRIDE, mask_prob=0.15)
    val_ds   = PaySimSequenceDataset(
        val_df, scaler=train_ds.scaler, context_len=context_len,
        train=False, feature_cols=FEATURE_COLS, stride=STRIDE)
    print(f"\nSequences — Train: {len(train_ds):,} | Val: {len(val_ds):,}")

    # Instantiate the underlying model and the focal-loss wrapper.
    model = LongRangeFraudTransformer(
        num_features  = len(train_ds.feature_cols),
        context_len   = context_len,
        num_type_cats = NUM_TYPE_CATS,
        type_emb_dim  = TYPE_EMB_DIM,
        type_col_idx  = TYPE_COL_IDX,
    )
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    wrapped = WrappedModel(model, fraud_weight=1.0, focal_gamma=2.0)

    # ── TrainingArguments ─────────────────────────────────────────────
    # Cosine LR schedule, ~15% warmup, AUPRC chooses the best checkpoint.
    # The try/except below covers the rename of `evaluation_strategy` →
    # `eval_strategy` between Transformers versions.
    warmup = int(0.15 * epochs * len(train_ds) // 64)
    try:
        args = TrainingArguments(
            output_dir                  = CKPT_DIR,
            num_train_epochs            = epochs,
            per_device_train_batch_size = 64,
            eval_strategy               = "epoch",
            save_strategy               = "epoch",
            learning_rate               = 5e-5,
            lr_scheduler_type           = "cosine",
            warmup_steps                = warmup,
            max_grad_norm               = 1.0,
            save_total_limit            = 2,
            remove_unused_columns       = False,
            load_best_model_at_end      = True,
            metric_for_best_model       = "auprc",
            greater_is_better           = True,
            report_to                   = "none",
        )
    except TypeError:
        # Older transformers — fall back to the legacy keyword name.
        args = TrainingArguments(
            output_dir                  = CKPT_DIR,
            num_train_epochs            = epochs,
            per_device_train_batch_size = 64,
            evaluation_strategy         = "epoch",
            save_strategy               = "epoch",
            learning_rate               = 5e-5,
            max_grad_norm               = 1.0,
            save_total_limit            = 2,
            remove_unused_columns       = False,
            load_best_model_at_end      = True,
            metric_for_best_model       = "auprc",
            greater_is_better           = True,
            report_to                   = "none",
        )

    def collate(batch):
        """Stack tensors per key — works because every item has the same shape."""
        return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}

    trainer = Trainer(
        model           = wrapped,
        args            = args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        data_collator   = collate,
        compute_metrics = compute_metrics,
    )

    print("\nStarting training...")
    trainer.train()

    # ── Save artefacts ────────────────────────────────────────────────
    # Saving the scaler and the label encoder is just as important as
    # saving the weights — deploy_model reuses both to keep features
    # numerically identical to what the model saw during training.
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), os.path.join(output_dir, "model_weights.pt"))

    with open(os.path.join(output_dir, "scaler.pkl"), "wb") as f:
        pickle.dump(train_ds.scaler, f)

    with open(os.path.join(output_dir, "label_encoder.pkl"), "wb") as f:
        pickle.dump(le, f)

    config = {
        "num_features":   len(train_ds.feature_cols),
        "feature_cols":   train_ds.feature_cols,
        "context_length": context_len,
        "stride":         STRIDE,
        "type_col_idx":   TYPE_COL_IDX,
        "num_type_cats":  NUM_TYPE_CATS,
        "type_emb_dim":   TYPE_EMB_DIM,
    }
    with open(os.path.join(output_dir, "config.json"), "w") as f:
        json.dump(config, f, indent=2)

    print(f"\n✓ All artifacts saved to {output_dir}/")
    print(f"  model_weights.pt  — {os.path.getsize(os.path.join(output_dir, 'model_weights.pt')):,} bytes")
    return model

## Run training
Edit `EPOCHS` and `CONTEXT_LEN` as needed, then run this cell.

In [ ]:
# ── Run training ────────────────────────────────────────────────────────
# 20 epochs / context length 20 was the sweet spot in the report — feel
# free to scale either down for a quick smoke test.
EPOCHS      = 20
CONTEXT_LEN = 20

trained_model = train(
    data_path   = DATA_CSV,
    output_dir  = MODEL_DIR,
    epochs      = EPOCHS,
    context_len = CONTEXT_LEN,
)
print("\nTraining complete. Model is saved.")